# Variables de rezago para `firms_spain_final.csv`

Este notebook crea variables meteorológicas retrospectivas para cada observación. Agrega las mediciones disponibles por **comunidad autónoma y fecha** y calcula los rezagos sobre días naturales. Así, los periodos previos excluyen el día de la observación.

**Importante:** el CSV contiene observaciones de incendios, no una tabla meteorológica diaria completa e independiente. Los promedios diarios regionales se calculan a partir de las filas disponibles del propio CSV. Los valores deben interpretarse como aproximaciones regionales basadas en esas observaciones; no como clima exacto en cada coordenada. Las ventanas se dejan como `NaN` si falta cualquiera de sus días.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Ruta del archivo encontrado en el escritorio.
csv_path = Path.home() / 'Desktop' / 'wildfires-project-development' / 'data' / 'processed' / 'enriched' / 'firms_spain_final.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'No se encontró el CSV: {csv_path}')

df = pd.read_csv(csv_path, low_memory=False)
df['acq_date'] = pd.to_datetime(df['acq_date'], errors='coerce').dt.normalize()
required = {'acq_date', 'ccaa', 'precipitation', 'temperature_2m'}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f'Faltan columnas necesarias: {sorted(missing)}')

print(f'{len(df):,} filas; fechas {df.acq_date.min().date()} a {df.acq_date.max().date()}')
print(f"Comunidades autónomas: {df['ccaa'].nunique(dropna=True)}")
df.head()

35,284 filas; fechas 2023-01-01 a 2024-12-31
Comunidades autónomas: 18


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,clc_class_W,clc_class_E,clc_class_NW,clc_class_NE,clc_class_SW,clc_class_SE,clc_uniform_surroundings,ccaa,sum_prevention,sum_extinction
0,43.1262,-4.0911,316.0,1.1,1.0,2023-01-01,221,Aqua,MODIS,92,...,3,3,3,3,3,3,False,Cantabria,6.50,13.30
1,39.2047,-1.7606,304.4,1.0,1.0,2023-01-01,1326,Aqua,MODIS,59,...,5,5,5,5,5,5,True,Castilla-La Mancha,134.00,155.00
2,37.9941,-4.3265,303.5,1.1,1.0,2023-01-01,1326,Aqua,MODIS,53,...,7,6,7,7,7,7,False,Andalucía,278.00,295.00
3,43.5401,-5.8281,307.3,1.0,1.0,2023-01-02,1105,Terra,MODIS,65,...,6,5,5,5,9,5,False,Asturias,66.00,129.00
4,42.3885,-2.7282,301.2,3.1,1.7,2023-01-03,1008,Terra,MODIS,45,...,5,5,5,5,5,5,True,La Rioja,6.84,13.03


## Definición de las variables

Se calculan precipitación acumulada en los 3, 7 y 15 días anteriores; temperatura media en los 3, 7 y 15 días anteriores; y rezagos de un día para otras variables meteorológicas disponibles. Las ventanas usan días naturales y son estrictas: requieren datos diarios para todo el periodo.

In [2]:
# Variables meteorológicas disponibles en el archivo.
weather_cols = [c for c in [
    'precipitation', 'rain', 'temperature_2m', 'relative_humidity_2m',
    'wind_speed_10m', 'wind_gusts_10m', 'vapour_pressure_deficit',
    'surface_pressure', 'soil_moisture_0_to_7cm', 'soil_moisture_7_to_28cm',
    'shortwave_radiation', 'et0_fao_evapotranspiration', 'boundary_layer_height'
] if c in df.columns]

# Para evitar que el texto u otros valores no numéricos contaminen los promedios.
for col in weather_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Una media por comunidad y día a partir de las filas disponibles.
daily = (df.dropna(subset=['acq_date', 'ccaa'])
         .groupby(['ccaa', 'acq_date'], as_index=False)[weather_cols]
         .mean())

date_min, date_max = df['acq_date'].min(), df['acq_date'].max()
feature_frames = []
for region, part in daily.groupby('ccaa', sort=False):
    part = part.set_index('acq_date').sort_index()
    # Reindexado a días naturales: los días sin observación permanecen ausentes.
    calendar = pd.date_range(date_min, date_max, freq='D')
    part = part.reindex(calendar)
    part.index.name = 'acq_date'
    features = pd.DataFrame(index=part.index)
    features['ccaa'] = region

    # Valor del día inmediatamente anterior (día t-1).
    for col in weather_cols:
        features[f'{col}_lag_1d'] = part[col].shift(1)

    # Precipitación acumulada en los N días anteriores, sin incluir el día t.
    if 'precipitation' in part:
        for n in (3, 7, 15):
            features[f'precipitation_sum_prev_{n}d'] = part['precipitation'].shift(1).rolling(n, min_periods=n).sum()
    # Temperatura media en los N días anteriores, sin incluir el día t.
    if 'temperature_2m' in part:
        for n in (3, 7, 15):
            features[f'temperature_2m_mean_prev_{n}d'] = part['temperature_2m'].shift(1).rolling(n, min_periods=n).mean()
    feature_frames.append(features.reset_index())

lag_features = pd.concat(feature_frames, ignore_index=True)
lag_cols = [c for c in lag_features.columns if c not in ('ccaa', 'acq_date')]
df_lags = df.merge(lag_features, on=['ccaa', 'acq_date'], how='left', validate='many_to_one')
print(f'Variables nuevas: {len(lag_cols)}')
df_lags[['acq_date', 'ccaa', 'precipitation', 'temperature_2m'] + lag_cols[:8]].head(10)

Variables nuevas: 19


,acq_date,ccaa,precipitation,temperature_2m,precipitation_lag_1d,rain_lag_1d,temperature_2m_lag_1d,relative_humidity_2m_lag_1d,wind_speed_10m_lag_1d,wind_gusts_10m_lag_1d,vapour_pressure_deficit_lag_1d,surface_pressure_lag_1d
0,2023-01-01,Cantabria,0.0,12.653500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-01,Castilla-La Mancha,0.0,15.485000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-01,Andalucía,0.0,17.718334,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-02,Asturias,0.0,11.950000,0.064762,0.064762,13.057214,86.826293,5.819212,16.788000,0.199518,986.952801
4,2023-01-03,La Rioja,0.0,6.254333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2023-01-03,Región de Murcia,0.0,15.236666,0.000000,0.000000,17.756666,34.825502,2.724781,10.763999,1.326964,999.578105
6,2023-01-03,Comunidad Valenciana,0.1,14.955833,0.000000,0.000000,16.365833,57.820268,3.376023,12.401999,0.833029,1018.296622
7,2023-01-03,Región de Murcia,0.0,15.899666,0.000000,0.000000,17.756666,34.825502,2.724781,10.763999,1.326964,999.578105
8,2023-01-03,Región de Murcia,0.0,17.515001,0.000000,0.000000,17.756666,34.825502,2.724781,10.763999,1.326964,999.578105
9,2023-01-03,Comunidad Valenciana,0.0,17.685000,0.000000,0.000000,16.365833,57.820268,3.376023,12.401999,0.833029,1018.296622


## Revisar cobertura y guardar

Las fechas iniciales y los periodos con días regionales ausentes tendrán valores nulos. Revise la cobertura antes de usar estas variables en un modelo. El siguiente bloque exporta una copia enriquecida junto al CSV original.

In [3]:
coverage = df_lags[lag_cols].notna().mean().sort_values(ascending=False).rename('proporcion_no_nula')
display(coverage.to_frame())

output_path = csv_path.with_name('firms_spain_final_with_lags.csv')
df_lags.to_csv(output_path, index=False)
print(f'Archivo enriquecido guardado en: {output_path}')

,proporcion_no_nula
precipitation_lag_1d,0.844689
rain_lag_1d,0.844689
temperature_2m_lag_1d,0.844689
relative_humidity_2m_lag_1d,0.844689
wind_speed_10m_lag_1d,0.844689
wind_gusts_10m_lag_1d,0.844689
vapour_pressure_deficit_lag_1d,0.844689
surface_pressure_lag_1d,0.844689
soil_moisture_0_to_7cm_lag_1d,0.844689
soil_moisture_7_to_28cm_lag_1d,0.844689


Archivo enriquecido guardado en: C:\Users\gonza\Desktop\wildfires-project-development\data\processed\enriched\firms_spain_final_with_lags.csv
